# SpaceX Falcon 9 - EDA with SQL

Load the SpaceX launch table into an in-memory SQLite database and answer questions with SQL.

In [1]:
import sqlite3
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [3]:
df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv')
df = df.rename(columns={'Time (UTC)': 'Time'})
df.columns = [col.strip() for col in df.columns]
print(list(df.columns))
df.head()

['Date', 'Time', 'Booster_Version', 'Launch_Site', 'Payload', 'PAYLOAD_MASS__KG_', 'Orbit', 'Customer', 'Mission_Outcome', 'Landing_Outcome']


,Date,Time,Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [4]:
# Date format in the source is DD-MM-YYYY; add an ISO date column for easy filtering
df['Date_ISO'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce').dt.strftime('%Y-%m-%d')
con = sqlite3.connect(':memory:')
df.to_sql('SPACEXTBL', con, index=False, if_exists='replace')

def q(sql):
    return pd.read_sql_query(sql, con)

q('SELECT COUNT(*) AS total_rows FROM SPACEXTBL')

,total_rows
0,101


### Task 1: Names of the unique launch sites

In [5]:
q('SELECT DISTINCT Launch_Site FROM SPACEXTBL')

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


### Task 2: 5 records where the launch site begins with 'CCA'

In [6]:
q("SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5")

,Date,Time,Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome,Date_ISO
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute),2010-04-06
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute),2010-08-12
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt,None
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt,2012-08-10
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt,2013-01-03


### Task 3: Total payload mass carried by boosters launched by NASA (CRS)

In [7]:
q("SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_kg FROM SPACEXTBL WHERE Customer LIKE 'NASA (CRS)%'")

,total_payload_kg
0,48213


### Task 4: Average payload mass carried by booster version F9 v1.1

In [8]:
q("SELECT AVG(PAYLOAD_MASS__KG_) AS avg_payload_kg FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%'")

,avg_payload_kg
0,2534.666667


### Task 5: Date of the first successful landing outcome on a ground pad

In [9]:
q("SELECT MIN(Date_ISO) AS first_ground_pad_success FROM SPACEXTBL WHERE Landing_Outcome LIKE 'Success (ground pad)%'")

,first_ground_pad_success
0,2017-01-05


### Task 6: Boosters with success on a drone ship and payload mass between 4000 and 6000 kg

In [10]:
q("SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE Landing_Outcome LIKE 'Success (drone ship)%' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000")

,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


### Task 7: Total number of successful and failed mission outcomes

In [11]:
q('SELECT Mission_Outcome, COUNT(*) AS total FROM SPACEXTBL GROUP BY Mission_Outcome')

,Mission_Outcome,total
0,Failure (in flight),1
1,Success,98
2,Success,1
3,Success (payload status unclear),1


### Task 8: Booster versions that carried the maximum payload mass

In [12]:
q('SELECT DISTINCT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)')

,Booster_Version,PAYLOAD_MASS__KG_
0,F9 B5 B1048.4,15600
1,F9 B5 B1049.4,15600
2,F9 B5 B1051.3,15600
3,F9 B5 B1056.4,15600
4,F9 B5 B1048.5,15600
5,F9 B5 B1051.4,15600
6,F9 B5 B1049.5,15600
7,F9 B5 B1060.2,15600
8,F9 B5 B1058.3,15600
9,F9 B5 B1051.6,15600


### Task 9: Failed drone ship landings in 2015 (month, booster version, launch site)

In [13]:
q("SELECT substr(Date_ISO, 6, 2) AS month, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTBL WHERE Landing_Outcome LIKE 'Failure (drone ship)%' AND substr(Date_ISO, 1, 4) = '2015'")

,month,Landing_Outcome,Booster_Version,Launch_Site
0,10,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40


### Task 10: Rank landing outcomes between 2010-06-04 and 2017-03-20 (descending count)

In [14]:
q("SELECT Landing_Outcome, COUNT(*) AS outcome_count FROM SPACEXTBL WHERE Date_ISO BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY outcome_count DESC")

,Landing_Outcome,outcome_count
0,No attempt,7
1,Success (ground pad),2
2,Success (drone ship),2
3,Failure (drone ship),2
4,Failure (parachute),1
5,Controlled (ocean),1
